# 04 – Deep Learning / NLP Training (Transformer Fine-tuning)

**Goals**
- Fine-tune a Transformer on PhreshPhish cleaned text (`input_text`)
- Focus on **low false-positive rate** (high Precision @ high Recall)
- Use Hugging Face `Trainer` + `datasets`
- Memory-efficient (gradient accumulation, fp16, early stopping)
- Export best model for the inference app

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
from pathlib import Path
from datasets import Dataset, DatasetDict, ClassLabel
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
import evaluate

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
## 1. Configuration

# --------------- CONFIG ---------------
MODEL_NAME = "distilbert-base-uncased"   # good speed/quality trade-off
# Alternatives: "microsoft/deberta-v3-base", "Alibaba-NLP/gte-base-en-v1.5"

MAX_LENGTH = 256
BATCH_SIZE = 16
GRAD_ACCUM = 2
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
FP16 = torch.cuda.is_available()

OUTPUT_DIR = Path("models/dl")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = Path("processed/phreshphish_processed.parquet")

In [ ]:
## 2. Load Processed Data

df = pd.read_parquet(DATA_PATH)
print("Full processed shape:", df.shape)

# Optional: use a subset if GPU memory is limited
# df = df.sample(n=20000, random_state=42)

print(df["label"].value_counts(normalize=True).round(3) * 100)

In [ ]:
## 3. Train / Validation Split

train_df, val_df = train_test_split(
    df,
    test_size=0.15,
    stratify=df["label_id"],
    random_state=42
)

print(f"Train: {len(train_df)} | Val: {len(val_df)}")

In [ ]:
## 4. Convert to Hugging Face Datasets

def df_to_hf(df):
    ds = Dataset.from_pandas(df[["input_text", "label_id"]].reset_index(drop=True))
    ds = ds.rename_column("label_id", "labels")
    ds = ds.cast_column("labels", ClassLabel(names=["benign", "phish"]))
    return ds

train_ds = df_to_hf(train_df)
val_ds   = df_to_hf(val_df)

raw_datasets = DatasetDict({
    "train": train_ds,
    "validation": val_ds
})

print(raw_datasets)

In [ ]:
## 5. Tokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples["input_text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False          # dynamic padding later
    )

tokenized_datasets = raw_datasets.map(
    tokenize_function,
    batched=True,
    remove_columns=["input_text"]
)

print(tokenized_datasets)

In [ ]:
## 6. Metrics (Low-FP oriented)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    
    ap  = average_precision_score(labels, probs)
    roc = roc_auc_score(labels, probs)
    
    # Precision at specific recalls
    precision, recall, _ = precision_recall_curve(labels, probs)
    
    def prec_at_recall(target):
        idx = np.argmin(np.abs(recall - target))
        return precision[idx]
    
    return {
        "average_precision": ap,
        "roc_auc": roc,
        "precision_at_90_recall": prec_at_recall(0.90),
        "precision_at_95_recall": prec_at_recall(0.95),
    }

In [ ]:
## 7. Model

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "benign", 1: "phish"},
    label2id={"benign": 0, "phish": 1}
)

# Optional: class weights (slight emphasis on phishing)
# You can also use focal loss later if needed

In [ ]:
## 8. Training Arguments

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    eval_strategy="epoch",               # ← changed from evaluation_strategy
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    fp16=FP16,
    load_best_model_at_end=True,
    metric_for_best_model="average_precision",
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=50,
    report_to="none",
    seed=42,
)

In [ ]:
## 9. Trainer

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [ ]:
## 10. Train

train_result = trainer.train()
print(train_result)

In [ ]:
## 11. Final Evaluation on Validation Set

metrics = trainer.evaluate()
print("\nValidation Metrics:")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

In [ ]:
## 12. Find good decision threshold (low FPR)

# Get probabilities on validation set
pred_output = trainer.predict(tokenized_datasets["validation"])
logits = pred_output.predictions
probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
labels = pred_output.label_ids

precision, recall, thresholds = precision_recall_curve(labels, probs)

def best_threshold(target_recall=0.92):
    idx = np.argmin(np.abs(recall - target_recall))
    return thresholds[idx], precision[idx], recall[idx]

thr, prec, rec = best_threshold(0.92)
print(f"Recommended threshold for ~92% Recall: {thr:.4f}")
print(f"Precision = {prec:.4f} | Recall = {rec:.4f}")

In [ ]:
## 13. Save final model + tokenizer + threshold

final_dir = OUTPUT_DIR / "best_model"
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)

# Save threshold and metadata
import json
meta = {
    "model_name": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "threshold": float(thr),
    "label2id": {"benign": 0, "phish": 1},
    "id2label": {0: "benign", 1: "phish"}
}
with open(final_dir / "meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print(f"Model saved to → {final_dir}")

In [ ]:
## 14. Quick inference test

from transformers import pipeline

clf = pipeline(
    "text-classification",
    model=str(final_dir),
    tokenizer=str(final_dir),
    device=0 if torch.cuda.is_available() else -1,
    top_k=None
)

test_texts = [
    "https://secure-login-paypal.com/verify [SEP] Please enter your credentials to continue",
    "https://www.wikipedia.org [SEP] Wikipedia The Free Encyclopedia"
]

for t in test_texts:
    print(clf(t[:512]))
    print("-" * 60)

## Summary

- Fine-tuned `distilbert-base-uncased` (changeable in config)
- Optimised for **Average Precision** and Precision@high-Recall
- Best model + tokenizer + decision threshold saved
- Ready for Gradio / FastAPI inference app

**Next** → `05_evaluation_benchmarks.ipynb` or directly build the app